# Part 1: Data cleaning

This notebook builds the one clean table every later notebook uses: the six systems (UAE, Saudi Arabia, Qatar, Jordan, Morocco, Palestine), the 17 predictors from our pitch, the 10 maths plausible values and the weights.

Cleaning here only applies **fixed rules** that don't learn anything from the data. Anything that learns from the data (filling blanks with a median, scaling, clipping at percentiles) happens later inside the training folds, so no test information leaks into training.

| | |
|---|---|
| Reads | `../data/parquet/` (student, school and timing files), `../data/data_dictionary.csv` |
| Writes | `../data/clean/pisa6_clean.parquet`, `../data/clean/columns.json` |

**Where each part comes from**

| Section | Based on |
|---|---|
| 1.1, 1.3, 1.5–1.8, 1.10–1.11 | Mohamed, `01_clean_table` (branch `preprocessing-and-EDA`) |
| 1.2, 1.3 join checks, 1.5 source checks | Annie, `03_eda_observations` sections 2–4 (branch `annie-eda`) |
| 1.4 | Fathah, `fathah_eda_cleaning` (branch `fathah-eda-cleaning`) |
| 1.9 | New: not in any team notebook |

# 1.1) Load the six systems
*From Mohamed, steps 1–2.*

We only read the columns we need and only the six systems. The student file has 1,278 columns, so loading everything would take several GB of memory.

In [1]:
from pathlib import Path
import json
import re
import numpy as np
import pandas as pd

DATA = Path("../data")
SYSTEMS = ["ARE", "SAU", "QAT", "JOR", "MAR", "PSE"]
LEVEL2 = 420.07   # lower bound of maths Level 2 in PISA 2022

# the 17 predictors from the pitch, in 5 factor groups
GROUPS = {
    "family": ["ESCS", "PAREDINT", "HISEI", "HOMEPOS"],
    "maths_attitudes": ["ANXMAT", "MATHEFF", "MATHEF21"],
    "classroom": ["DISCLIM", "TEACHSUP"],
    "climate_resources": ["BELONG", "BULLIED", "EDUSHORT", "STAFFSHORT"],
    "school_org": ["MCLSIZE", "STRATIO", "SCHLTYPE", "SCHSIZE"],
}
SCHOOL_COLS = ["EDUSHORT", "STAFFSHORT", "MCLSIZE", "STRATIO", "SCHLTYPE", "SCHSIZE"]
STUDENT_COLS = [c for g in GROUPS.values() for c in g if c not in SCHOOL_COLS]

PV = [f"PV{i}MATH" for i in range(1, 11)]
REP = [f"W_FSTURWT{i}" for i in range(1, 81)]
EDA_COLS = ["ST004D01T", "GRADE", "IMMIG", "REPEAT", "EFFORT1", "ST273Q06JA", "ST273Q07JA"]
CANDIDATES = ["FAMSUP", "FEELSAFE", "ICTDISTR"]   # looked at in 1.4, not kept
ANX_ITEMS = [f"ST292Q0{i}JA" for i in range(1, 7)]   # maths anxiety items, used in 1.9

stu = pd.read_parquet(DATA / "parquet/CY08MSP_STU_QQQ",
                      columns=["CNT", "CNTSCHID", "CNTSTUID"] + STUDENT_COLS + PV + ["W_FSTUWT", "SENWT"] + REP
                              + EDA_COLS + CANDIDATES + ANX_ITEMS,
                      filters=[("CNT", "in", SYSTEMS)])
sch = pd.read_parquet(DATA / "parquet/CY08MSP_SCH_QQQ",
                      columns=["CNT", "CNTSCHID"] + SCHOOL_COLS + ["TOTAT", "SC002Q01TA", "SC002Q02TA"],
                      filters=[("CNT", "in", SYSTEMS)])

# IDs are stored as decimals (78400371.0), make them whole numbers
assert (stu["CNTSTUID"] % 1 == 0).all() and (stu["CNTSCHID"] % 1 == 0).all()
stu[["CNTSCHID", "CNTSTUID"]] = stu[["CNTSCHID", "CNTSTUID"]].astype("int64")
sch["CNTSCHID"] = sch["CNTSCHID"].astype("int64")

log = [("1.1 load", len(stu), stu.shape[1])]
print("students:", stu.shape, " schools:", sch.shape)
print("duplicate students:", stu.duplicated(["CNT", "CNTSTUID"]).sum())
stu.groupby("CNT").agg(students=("CNTSTUID", "size"), schools=("CNTSCHID", "nunique")).loc[SYSTEMS]

students: (61775, 122)  schools: (1973, 11)
duplicate students: 0


,students,schools
CNT,,
ARE,24600,840
SAU,6928,193
QAT,7676,229
JOR,7799,260
MAR,6867,178
PSE,7905,273


# 1.2) Hidden missing codes
*From Annie, section 2.*

PISA codes missing answers as 95/97/98/99 (995–999 for some school variables). Our Parquet files should already have these as `NaN`. We check every predictor against its own codes in the data dictionary, so a code like 99 can't slip through as a real value.

In [2]:
dd = pd.read_csv(DATA / "data_dictionary.csv")
MISSING_LABELS = {"valid skip", "not applicable", "invalid", "no response"}

rows = []
for source, table, cols in [("student", stu, STUDENT_COLS), ("school", sch, SCHOOL_COLS)]:
    for col in cols:
        labels = json.loads(dd.query("dataset == @source and variable == @col")["value_labels"].iloc[0])
        codes = [float(k) for k, v in labels.items() if v.lower() in MISSING_LABELS]
        rows.append({"variable": col, "missing_codes": codes,
                     "codes_still_in_data": table[col].isin(codes).sum(), "already_NaN": table[col].isna().sum()})

codes_df = pd.DataFrame(rows)
assert codes_df["codes_still_in_data"].sum() == 0
codes_df

,variable,missing_codes,codes_still_in_data,already_NaN
0,ESCS,"[95.0, 97.0, 98.0, 99.0]",0,2432
1,PAREDINT,"[9995.0, 9997.0, 9998.0, 9999.0]",0,2659
2,HISEI,"[995.0, 997.0, 998.0, 999.0]",0,9017
3,HOMEPOS,"[95.0, 97.0, 98.0, 99.0]",0,1764
4,ANXMAT,"[95.0, 97.0, 98.0, 99.0]",0,14123
5,MATHEFF,"[95.0, 97.0, 98.0, 99.0]",0,13037
6,MATHEF21,"[95.0, 97.0, 98.0, 99.0]",0,13837
7,DISCLIM,"[95.0, 97.0, 98.0, 99.0]",0,10455
8,TEACHSUP,"[95.0, 97.0, 98.0, 99.0]",0,11140
9,BELONG,"[95.0, 97.0, 98.0, 99.0]",0,5714


> **Note:** no missing code is left in any of the 17 predictors. Every blank is already `NaN`, so nothing needs recoding.

# 1.3) Join the school data onto students
*From Mohamed, step 3, with Annie's join checks.*

Many students share one school, so this is a many-to-one join on country **and** school ID. A left join keeps every student even if their school sent no questionnaire.

In [3]:
assert not stu.duplicated(["CNT", "CNTSTUID"]).any()
assert not sch.duplicated(["CNT", "CNTSCHID"]).any()

df = stu.merge(sch, on=["CNT", "CNTSCHID"], how="left", validate="many_to_one", indicator=True)
assert len(df) == len(stu)
print(df["_merge"].value_counts().to_string())
df = df.drop(columns="_merge")

# every school value should arrive unchanged
joined = df.drop_duplicates(["CNT", "CNTSCHID"]).set_index(["CNT", "CNTSCHID"])[SCHOOL_COLS].sort_index()
pd.testing.assert_frame_equal(joined, sch.set_index(["CNT", "CNTSCHID"])[SCHOOL_COLS].sort_index())

# students whose school answered none of the six school questions
df["SCH_NODATA"] = df[SCHOOL_COLS].isna().all(axis=1).astype(int)
log.append(("1.3 join", len(df), df.shape[1]))
df.groupby("CNT")["SCH_NODATA"].agg(["sum", "mean"]).loc[SYSTEMS].round(3)

_merge
both          61775
left_only         0
right_only        0


,sum,mean
CNT,,
ARE,2424,0.099
SAU,0,0.000
QAT,0,0.000
JOR,0,0.000
MAR,0,0.000
PSE,0,0.000


> **Note:** every student found their school and no rows were duplicated. But 2,424 UAE students (9.9%) come from schools that sent back an empty questionnaire. No other system has this. `SCH_NODATA` marks them.

# 1.4) Candidate predictors we leave out
*From Fathah.*

Fathah also looked at family support (`FAMSUP`), feeling safe (`FEELSAFE`) and ICT distraction (`ICTDISTR`). Before adding anything to the 17 we check how much of each is missing per system.

In [4]:
df[CANDIDATES].isna().groupby(df["CNT"]).mean().mul(100).round(1).loc[SYSTEMS]

,FAMSUP,FEELSAFE,ICTDISTR
CNT,,,
ARE,18.8,9.3,100.0
SAU,19.0,6.0,49.6
QAT,34.4,12.7,100.0
JOR,32.9,9.8,40.7
MAR,45.2,4.8,35.9
PSE,24.8,8.8,100.0


> **Note:** `ICTDISTR` is 100% missing in the UAE, Qatar and Palestine, so it can't be used in a six-system model (section 2.3 shows why: those systems didn't run the ICT questionnaire). `FAMSUP` is missing for 19–45% of students. We keep the 17 predictors from the pitch and drop the three candidates.

In [5]:
df = df.drop(columns=CANDIDATES)

# 1.5) Impossible values
*From Mohamed, step 5, with Annie's source checks.*

The student-teacher ratio `STRATIO` is school size ÷ number of teachers, capped by the OECD at 1 and 100. First we check that this is really how it was built, using the raw enrolment and teacher counts.

In [6]:
s = sch.dropna(subset=["SCHSIZE", "TOTAT", "STRATIO"]).query("TOTAT > 0")
rebuilt = (s["SCHSIZE"] / s["TOTAT"]).clip(1, 100)
print("STRATIO = SCHSIZE / TOTAT, capped 1-100:", f"{np.isclose(rebuilt, s['STRATIO'], atol=0.01).mean():.1%} of {len(s)} schools")

e = sch.dropna(subset=["SC002Q01TA", "SC002Q02TA", "SCHSIZE"])
print("boys + girls = SCHSIZE:", f"{(e['SC002Q01TA'] + e['SC002Q02TA'] == e['SCHSIZE']).mean():.1%} of {len(e)} schools")

STRATIO = SCHSIZE / TOTAT, capped 1-100: 100.0% of 1690 schools
boys + girls = SCHSIZE: 100.0% of 1663 schools


Both checks pass, so odd ratios come from odd inputs, not a calculation error. Some are impossible for a secondary school:

- **ratio below 2**: fewer students than teachers (e.g. 42 students, 67 teachers). The school size is wrong, so `STRATIO` and `SCHSIZE` both become `NaN`.
- **ratio above 50**: far too few teachers (e.g. 8,000 students, 2 teachers). The teacher count is wrong, so only `STRATIO` becomes `NaN`.

The original values are kept as `STRATIO_raw` and `SCHSIZE_raw`. The rule was set before looking at maths scores.

In [7]:
df["STRATIO_raw"], df["SCHSIZE_raw"] = df["STRATIO"], df["SCHSIZE"]
too_low, too_high = df["STRATIO"] < 2, df["STRATIO"] > 50
df.loc[too_low | too_high, "STRATIO"] = np.nan
df.loc[too_low, "SCHSIZE"] = np.nan

flagged = (df[too_low | too_high].groupby(["CNT", "CNTSCHID"])
           .agg(STRATIO=("STRATIO_raw", "first"), SCHSIZE=("SCHSIZE_raw", "first"),
                teachers=("TOTAT", "first"), students=("CNTSTUID", "size")))
print(len(flagged), "schools flagged,", (too_low | too_high).sum(), "students")
df = df.drop(columns=["TOTAT", "SC002Q01TA", "SC002Q02TA"])
log.append(("1.5 impossible values", len(df), df.shape[1]))
flagged.sort_values("STRATIO")

35 schools flagged, 1053 students


STRATIO  SCHSIZE  teachers  students
CNT CNTSCHID                                       
JOR 40000088    1.0000     42.0      67.0        30
QAT 63400056    1.0000     48.0      80.0        44
SAU 68200124    1.0000     14.0      33.0        12
MAR 50400099    1.0000      1.0       8.5         1
SAU 68200044    1.0000     25.0      29.0        23
    68200153    1.0000     53.0      59.0        41
    68200142    1.0000     10.0      21.5         7
ARE 78400371    1.0000     15.0      72.0        13
    78400725    1.0000     16.0      65.0        12
PSE 27500071    1.0606     35.0      33.0        35
SAU 68200136    1.1892     22.0      18.5        12
ARE 78400548    1.3750     55.0      40.0        41
SAU 68200191    1.3947     53.0      38.0        42
    68200070    1.4286     50.0      35.0        36
    68200082    1.4324     53.0      37.0        42
PSE 27500206    1.4583     35.0      24.0        33
SAU 68200100    1.4583     35.0      24.0        13
    68200147    1.5000     63.0      42.0        34
    68200135    1.5758     52.0      33.0        41
ARE 78400280    1.6216     60.0      37.0        28
SAU 68200071    1.6571     58.0      35.0        35
    68200117    1.7059     29.0      17.0        20
ARE 78400716    1.7667     53.0      30.0        40
MAR 50400173    1.8594    119.0      64.0        42
ARE 78400784    1.9111     86.0      45.0        82
    78400127   50.5120   6314.0     125.0        40
MAR 50400141   50.5625   1618.0      32.0        38
JOR 40000236   51.1765    870.0      17.0        22
ARE 78400082   53.0588   6314.0     119.0        39
MAR 50400069   53.8621    781.0      14.5        42
SAU 68200161   59.1111    532.0       9.0        13
QAT 63400048   65.0000    195.0       3.0         2
SAU 68200065   92.0000    138.0       1.5        32
    68200150  100.0000   8000.0       2.0        41
JOR 40000161  100.0000   2007.0      20.0        25

# 1.6) Column types
*From Mohamed, step 6.*

`SCHLTYPE` is a label, not an amount: 1 = private independent, 2 = private government-dependent, 3 = public. Type 2 is too rare to learn from, so we merge 1 and 2 into one binary column `SCHL_PRIVATE`. `MCLSIZE` stores the midpoint of each class-size band, so it can stay numeric.

In [8]:
print(pd.crosstab(df["CNT"], df["SCHLTYPE"]).loc[SYSTEMS], "\n")
df["SCHL_PRIVATE"] = df["SCHLTYPE"].map({1: 1, 2: 1, 3: 0})
print("MCLSIZE codes:", sorted(df["MCLSIZE"].dropna().astype(int).unique().tolist()))

SCHLTYPE    1.0  2.0   3.0
CNT                       
ARE       14032  205  7900
SAU        1244    0  5684
QAT        3598  171  3907
JOR        1308    0  6491
MAR         595    0  6272
PSE         376   51  7478 

MCLSIZE codes: [13, 18, 23, 28, 33, 38, 43, 48, 53]


# 1.7) Skewed columns
*From Mohamed, step 7.*

School size has a long right tail, so a few huge schools would dominate linear models and the MLP. A log fixes that.

In [9]:
df["SCHSIZE_log"] = np.log(df["SCHSIZE"])
numeric = [c for g in GROUPS.values() for c in g if c != "SCHLTYPE"] + ["SCHSIZE_log"]
df[numeric].skew().round(2).to_frame("skew").T

,ESCS,PAREDINT,HISEI,HOMEPOS,ANXMAT,MATHEFF,MATHEF21,DISCLIM,TEACHSUP,BELONG,BULLIED,EDUSHORT,STAFFSHORT,MCLSIZE,STRATIO,SCHSIZE,SCHSIZE_log
skew,-0.81,-1.9,-0.42,0.12,-0.13,-0.06,-0.01,-0.12,-0.63,1.01,0.8,0.51,0.58,0.5,1.09,3.1,-0.12


> **Note:** school size drops from a skew of 3.1 to about 0. `PAREDINT` (-1.9) is skewed because over half the students are at the top of the scale (16 years of schooling), which is a ceiling, not something a log can fix. `BELONG` and `STRATIO` are only just past 1, so we leave them.

# 1.8) Targets
*From Mohamed, step 8.*

PISA gives 10 plausible values per student, not one score. We keep all 10 and make a below-Level-2 flag for each: the regression target is `PVkMATH`, the classification target is `BELOW_L2_PVk`. Models run once per plausible value and the 10 results are combined.

In [10]:
BELOW = [f"BELOW_L2_PV{i}" for i in range(1, 11)]
for pv, below in zip(PV, BELOW):
    df[below] = (df[pv] < LEVEL2).astype(int)

assert df[PV].notna().all().all()
flips = df[BELOW].nunique(axis=1).gt(1).mean()
print(f"students whose below-Level-2 label changes across the 10 PVs: {flips:.1%}")
log.append(("1.8 targets", len(df), df.shape[1]))

students whose below-Level-2 label changes across the 10 PVs: 32.2%


> **Note:** the label flips for about a third of students. Using only `PV1MATH` would hide this uncertainty.

# 1.9) Response-quality flags
*New: not in any team notebook.*

Questionnaire answers are only useful if students actually read the questions. We use the timing file and the test-effort question to flag, not delete:

- `REACHED_ATTITUDES`: the student opened the maths-attitude screens (some stop before them)
- `SPEEDING`: under 1 second per item on the maths-anxiety screen
- `STRAIGHTLINE`: the same answer to every anxiety item
- `LOW_EFFORT`: 4 or less out of 10 on "how much effort did you put into this test"
- `ICT_MODULE`: the system ran the optional ICT questionnaire

In [11]:
# questionnaire screens in the order students see them (the dictionary keeps file order)
screens = [v for v in dd.query("dataset == 'timing'")["variable"]
           if v.startswith("ST") and v.endswith("_TT") and not v.startswith("STIntro")]
tim = pd.read_parquet(DATA / "parquet/CY08MSP_STU_TIM", columns=["CNT", "CNTSTUID", "Option_ICTQ"] + screens,
                      filters=[("CNT", "in", SYSTEMS)])
tim["CNTSTUID"] = tim["CNTSTUID"].astype("int64")

# last screen each student opened
opened = tim[screens].notna().to_numpy()
tim["last_screen"] = np.where(opened.any(axis=1), len(screens) - 1 - opened[:, ::-1].argmax(axis=1), -1)
first_attitude = min(screens.index(s) for s in ["ST290_TT", "ST291_TT", "ST292_TT"])
print(f"attitude screens start at screen {first_attitude} of {len(screens)}")

df = df.merge(tim[["CNT", "CNTSTUID", "Option_ICTQ", "last_screen", "ST292_TT"]], on=["CNT", "CNTSTUID"],
              how="left", validate="one_to_one")
df["REACHED_ATTITUDES"] = (df["last_screen"] >= first_attitude).astype(int)
df["ICT_MODULE"] = (df["Option_ICTQ"] == 1).astype(int)

# each student only sees 5 of the 6 anxiety items (they are rotated)
answered = df[ANX_ITEMS].notna().sum(axis=1)
print("anxiety items answered:", answered.value_counts().sort_index().to_dict())
full = answered >= 5
df["STRAIGHTLINE"] = (full & (df[ANX_ITEMS].nunique(axis=1) == 1)).astype(float).where(full)
df["SPEEDING"] = (df["ST292_TT"] / 1000 / answered < 1).astype(float).where(full & df["ST292_TT"].notna())
df["LOW_EFFORT"] = (df["EFFORT1"] <= 4).astype(float).where(df["EFFORT1"].notna())

df = df.drop(columns=ANX_ITEMS + ["Option_ICTQ", "last_screen", "ST292_TT"])
log.append(("1.9 quality flags", len(df), df.shape[1]))
df.groupby("CNT")[["REACHED_ATTITUDES", "ICT_MODULE", "STRAIGHTLINE", "SPEEDING", "LOW_EFFORT"]].mean().mul(100).round(1).loc[SYSTEMS]

attitude screens start at screen 62 of 92
anxiety items answered: {0: 13293, 1: 633, 2: 197, 3: 302, 4: 1469, 5: 45881}


,REACHED_ATTITUDES,ICT_MODULE,STRAIGHTLINE,SPEEDING,LOW_EFFORT
CNT,,,,,
ARE,96.0,0.0,38.2,9.3,5.6
SAU,98.8,100.0,39.7,3.4,2.8
QAT,96.4,0.0,38.2,11.3,6.7
JOR,99.1,100.0,29.5,11.3,6.0
MAR,91.4,100.0,25.9,2.7,3.5
PSE,98.8,0.0,27.4,9.5,4.7


> **Note:** these are flags only. Section 2.4 checks whether flagged students score differently before we decide anything. Two design facts came out of this: the anxiety items are rotated (each student sees 5 of 6), and only Saudi Arabia, Jordan and Morocco ran the ICT questionnaire.

# 1.10) Leakage check
*From Mohamed, step 9.*

Reading and science scores, the maths sub-scales and the raw test answers are almost copies of the target, so we never loaded them. This cell makes sure none of them, or any ID, weight or target, ended up as a feature.

In [12]:
FEATURES_BY_GROUP = {g: [{"SCHLTYPE": "SCHL_PRIVATE", "SCHSIZE": "SCHSIZE_log"}.get(c, c) for c in cols]
                     for g, cols in GROUPS.items()}
FEATURES = [c for cols in FEATURES_BY_GROUP.values() for c in cols]

assert not [c for c in df.columns if re.match(r"^PV\d+(READ|SCIE|MC[A-Z]{2}|MP[A-Z]{2})$", c)]
assert not set(FEATURES) & set(["CNT", "CNTSCHID", "CNTSTUID", "W_FSTUWT", "SENWT"] + PV + BELOW + REP)
assert len(FEATURES) == 17
FEATURES_BY_GROUP

{'family': ['ESCS', 'PAREDINT', 'HISEI', 'HOMEPOS'],
 'maths_attitudes': ['ANXMAT', 'MATHEFF', 'MATHEF21'],
 'classroom': ['DISCLIM', 'TEACHSUP'],
 'climate_resources': ['BELONG', 'BULLIED', 'EDUSHORT', 'STAFFSHORT'],
 'school_org': ['MCLSIZE', 'STRATIO', 'SCHL_PRIVATE', 'SCHSIZE_log']}

# 1.11) Missing flags and save
*From Mohamed, step 10.*

We keep every student. Each feature gets a 0/1 `_missing` column, since section 2.3 shows that a blank answer itself goes with lower scores. The two digital-distraction answers become 0/1 flags for the side question (1 = distracted in most or every maths lesson).

In [13]:
for c in FEATURES:
    df[f"{c}_missing"] = df[c].isna().astype(int)
for new, item in [("DISTRACTED_OWN", "ST273Q06JA"), ("DISTRACTED_OTHERS", "ST273Q07JA")]:
    df[new] = (df[item] <= 2).astype(float).where(df[item].notna())

roles = {
    "ids": ["CNT", "CNTSCHID", "CNTSTUID"],
    "targets": {"regression": PV, "classification": BELOW, "level2_cutoff": LEVEL2},
    "feature_groups": FEATURES_BY_GROUP,
    "missing_indicators": {g: [f"{c}_missing" for c in cols] for g, cols in FEATURES_BY_GROUP.items()},
    "flags": ["SCH_NODATA"],
    "weights": {"final": "W_FSTUWT", "senate": "SENWT", "replicates": REP},
    "side_questions": ["REPEAT", "ST273Q06JA", "ST273Q07JA", "DISTRACTED_OWN", "DISTRACTED_OTHERS"],
    "quality_flags": ["REACHED_ATTITUDES", "ICT_MODULE", "STRAIGHTLINE", "SPEEDING", "LOW_EFFORT"],
    "eda_only": ["ST004D01T", "GRADE", "IMMIG", "EFFORT1", "SCHLTYPE", "SCHSIZE"],
    "audit_only": ["STRATIO_raw", "SCHSIZE_raw"],
}

# every column must have exactly one role
listed = (roles["ids"] + PV + BELOW + FEATURES + sum(roles["missing_indicators"].values(), []) + roles["flags"]
          + ["W_FSTUWT", "SENWT"] + REP + roles["side_questions"] + roles["quality_flags"]
          + roles["eda_only"] + roles["audit_only"])
assert sorted(listed) == sorted(df.columns)

(DATA / "clean").mkdir(exist_ok=True)
df.to_parquet(DATA / "clean/pisa6_clean.parquet", index=False)
(DATA / "clean/columns.json").write_text(json.dumps(roles, indent=2))
pd.testing.assert_frame_equal(pd.read_parquet(DATA / "clean/pisa6_clean.parquet"), df)
log.append(("1.11 save", len(df), df.shape[1]))
print("saved", df.shape)

saved (61775, 158)


# 1.12) Cleaning summary

In [14]:
pd.DataFrame(log, columns=["step", "rows", "columns"])

,step,rows,columns
0,1.1 load,61775,122
1,1.3 join,61775,132
2,1.5 impossible values,61775,128
3,1.8 targets,61775,140
4,1.9 quality flags,61775,139
5,1.11 save,61775,158


- **61,775 students in 1,973 schools**, none dropped. Every row of the raw six-system sample is in the clean table.
- **No hidden missing codes**; all blanks are `NaN`.
- **35 schools** had an impossible student-teacher ratio. Their values are set to `NaN` and the originals kept.
- **Three candidate predictors dropped** (`ICTDISTR`, `FAMSUP`, `FEELSAFE`): too much missing, or not in the pitch.
- **Targets**: 10 scores and 10 below-Level-2 flags. A third of students change label across plausible values.
- **New quality flags** for not reaching the attitude screens, speeding, straightlining, low effort and the ICT module.

Next: `02_eda.ipynb`.